# ADR Evaluation (clean, fast, consistent)

This notebook is a simpler replacement for `ADR_Eval.ipynb`.

It does:
1) **Rollout evaluation** (absolute + relative errors), batched.
2) **Intrinsic metrics** along trajectories:
   - Dynamics Jacobian singular values (∂f/∂z)
   - Decoder gain along trajectories (approx spectral norm of J_dec)
   - Latent error (z_pred vs z_true = enc(u_true))
3) **Paired comparisons** between methods (same windows).

**Consistency rules**:
- `umin/umax` are computed on **training μ and training-time** (coarse) and reused everywhere.
- Evaluation defaults to **fine** (`u_fine`, `t_fine`) but can be switched.

In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.dataset import norm_u, inv_norm_u, H5Snapshots
from src.node import integrate_latent_eval

In [2]:
experiments = {
    "vanilla": {
        "lam_reg": 0
    },
    "stoch_iso": {
        "lam_reg": 0.1
    },    
    "operator_norm": {
        "lam_reg": 0.1
    },
    "curvature": {
        "lam_reg": 1.0,
    },
    "stiefel": {
        "lam_reg": 0
    }
}

In [29]:
# -----------------
# USER CONFIG
# -----------------
H5_PATH    = "adr_full.h5" 
SPLIT      = "extrap"           # "train" | "interp" | "extrap"
EVAL_FIELD = "u_fine"           # "u_fine" (recommended) or "u_coarse"
RK_METHOD  = "rk2"              # "rk2" or "rk4"

HORIZON    = 320                # in indices of EVAL_FIELD grid
N_MU       = 100
N_STARTS   = 10
BATCH      = 32

V          = 1                 # AE version
SELECTION  = "best_swa"        # ode_{selection}.pt   best_swa | target_swa | last_swa
METHODS    = [x for x in experiments.keys()]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32

# intrinsic metric cost controls
INTR_STEPS = 6
INTR_MAX_WINDOWS = 80
DEC_GAIN_ITERS = 5

SEED = 0
rng = np.random.default_rng(SEED)

print("DEVICE:", DEVICE)

DEVICE: cuda


In [30]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import copy

# --------- CONFIG ----------
TOP_K = 3

# If you want to restrict versions/runs/methods, set these; otherwise we auto-discover.
AE_VERSIONS = [1,2]; NODE_RUNS   =  range(1,16+1)

# "Robust worst-best" target selection:
# q = 1.0  -> max(best_vals)  (strict "worst best")
# q = 0.9  -> 90th percentile (more robust to one outlier)
TARGET_Q = 1.0

OUTPUT_ROOT = Path("output/NODE")
MAP_LOCATION = "cuda"


In [31]:
def parse_method_version_run(stats_path: Path):
    # stats_path: output/NODE/{method}_v{version}/run_{run}/stats.json
    run_dir = stats_path.parent
    method_v = run_dir.parent.name          # "{method}_v{version}"
    run_name = run_dir.name                 # "run_{run}"

    if "_v" not in method_v:
        raise ValueError(f"Cannot parse version from: {method_v}")
    method, v_str = method_v.rsplit("_v", 1)
    version = int(v_str)

    if not run_name.startswith("run_"):
        raise ValueError(f"Cannot parse run from: {run_name}")
    run = int(run_name.split("_", 1)[1])

    return method, version, run, run_dir

# Discover all stats.json files
stats_files = sorted(OUTPUT_ROOT.glob("*_v*/run_*/stats.json"))
print("Found stats.json:", len(stats_files))

records = []
stats_map = {}  # (version, method, run) -> dict(stats) + run_dir

for p in stats_files:
    method, version, run, run_dir = parse_method_version_run(p)

    if AE_VERSIONS is not None and version not in AE_VERSIONS:
        continue
    if NODE_RUNS is not None and run not in NODE_RUNS:
        continue

    with open(p, "r") as f:
        st = json.load(f)

    # basic sanity
    if "mse_val" not in st or "mse_train" not in st:
        print(f"[skip] missing mse_train/mse_val in {p}")
        continue

    stats_map[(version, method, run)] = {"stats": st, "run_dir": run_dir}

print("Loaded runs:", len(stats_map))
# list(stats_map.keys())


Found stats.json: 160
Loaded runs: 160


In [32]:
best_vals = []
for (version, method, run), obj in stats_map.items():
    val = np.asarray(obj["stats"]["mse_val"], dtype=float)
    best_vals.append(val.min())

best_vals = np.asarray(best_vals, dtype=float)
if len(best_vals) == 0:
    raise RuntimeError("No stats loaded; check OUTPUT_ROOT or filters.")

target_val = float(np.quantile(best_vals, TARGET_Q))
print(f"Target val MSE: (q={TARGET_Q}): {target_val:.6e}")
print(f"Best-val range: {float(best_vals.min()):.6e} to {float(best_vals.max()):.6e}")


Target val MSE: (q=1.0): 2.715394e-03
Best-val range: 4.567359e-04 to 2.715394e-03


In [33]:
def first_epoch_runningbest_leq(val_mse: np.ndarray, target: float):
    """
    Return 1-based epoch where running best first reaches <= target.
    If never reaches, return None.
    """
    rb = np.minimum.accumulate(val_mse)
    hits = np.where(rb <= target)[0]
    if len(hits) == 0:
        return None
    return int(hits[0] + 1)

def trailing_epochs(ep: int, k: int):
    """Return [max(1, ep-k+1), ..., ep] (1-based)."""
    start = max(1, ep - k + 1)
    return list(range(start, ep + 1))

def load_ckpt_model(run_dir: Path, epoch: int):
    ckpt_path = run_dir / f"ode_{epoch}.pt"
    if not ckpt_path.exists():
        return None, ckpt_path
    model = torch.load(ckpt_path, map_location=MAP_LOCATION, weights_only=False)
    model.eval()
    return model, ckpt_path

def swa_average_checkpoints(run_dir: Path, epochs: list[int], save_path: Path):
    """
    Uniform average of floating tensors over checkpoints in `epochs`.
    Non-floating tensors are copied from the LAST loaded checkpoint.
    Saves a torch model at save_path.
    Returns: (used_epochs, save_path) or ([], None) if nothing loaded.
    """
    used = []
    sum_sd = None
    last_sd = None
    template_model = None

    for ep in epochs:
        model, ckpt_path = load_ckpt_model(run_dir, ep)
        if model is None:
            print(f"[warn] missing {ckpt_path}")
            continue

        sd = model.state_dict()
        if sum_sd is None:
            template_model = copy.deepcopy(model)
            sum_sd = {k: v.detach().clone().to(MAP_LOCATION) for k, v in sd.items()}
        else:
            for k in sum_sd:
                if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
                    sum_sd[k].add_(sd[k].detach().to(MAP_LOCATION, dtype=sum_sd[k].dtype))
                else:
                    # keep placeholder; we'll overwrite from last checkpoint below
                    pass

        last_sd = {k: v.detach().to(MAP_LOCATION) for k, v in sd.items()}
        used.append(ep)

    if not used:
        print(f"[skip] no checkpoints found for {run_dir}")
        return [], None

    # divide floating tensors
    n = float(len(used))
    for k in sum_sd:
        if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
            sum_sd[k].div_(n)
        else:
            # copy non-floating from last checkpoint
            sum_sd[k] = last_sd[k].clone()

    template_model.load_state_dict(sum_sd, strict=True)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(template_model, save_path)
    return used, save_path


In [34]:
rows = []

for (version, method, run), obj in sorted(stats_map.items()):
    st = obj["stats"]
    run_dir: Path = obj["run_dir"]

    train = np.asarray(st["mse_train"], dtype=float)
    val   = np.asarray(st["mse_val"], dtype=float)
    n_epochs = len(val)

    # epochs (1-based)
    best_train_ep = int(np.argmin(train) + 1)
    best_val_ep   = int(np.argmin(val) + 1)

    # "target reached" epoch using running best
    target_ep = first_epoch_runningbest_leq(val, target_val)
    target_reached = target_ep is not None
    if target_ep is None:
        target_ep = n_epochs  # fallback: use last epoch if never reached

    # trailing windows
    epochs_target = trailing_epochs(target_ep, TOP_K)
    epochs_best   = trailing_epochs(best_val_ep, TOP_K)
    epochs_last   = trailing_epochs(n_epochs, TOP_K)

    # save SWA models
    used_t, p_t = swa_average_checkpoints(run_dir, epochs_target, run_dir / "ode_target_swa.pt")
    used_b, p_b = swa_average_checkpoints(run_dir, epochs_best,   run_dir / "ode_best_swa.pt")
    used_l, p_l = swa_average_checkpoints(run_dir, epochs_last,   run_dir / "ode_last_swa.pt")

    rows.append({
        "version": version,
        "method": method,
        "run": run,
        "n_epochs": n_epochs,

        "best_train": float(train.min()),
        "best_val": float(val.min()),
        "best_train_ep": best_train_ep,
        "best_val_ep": best_val_ep,

        "target_val": float(target_val),
        "target_ep": int(target_ep),
        "target_reached": bool(target_reached),

        "std_train": float(train.std(ddof=0)),
        "std_val": float(val.std(ddof=0)),

        "target_swa_epochs": used_t,
        "best_swa_epochs": used_b,
        "last_swa_epochs": used_l,
    })

metadata_df = pd.DataFrame(rows).sort_values(["version", "method", "run"]).reset_index(drop=True)
metadata_df


,version,method,run,n_epochs,best_train,best_val,best_train_ep,best_val_ep,target_val,target_ep,target_reached,std_train,std_val,target_swa_epochs,best_swa_epochs,last_swa_epochs
0,1,curvature,1,50,0.000795,0.001303,50,38,0.002715,13,True,0.024328,0.019597,"[11, 12, 13]","[36, 37, 38]","[48, 49, 50]"
1,1,curvature,2,50,0.000870,0.001077,49,50,0.002715,13,True,0.024476,0.016335,"[11, 12, 13]","[48, 49, 50]","[48, 49, 50]"
2,1,curvature,3,50,0.000893,0.001193,49,50,0.002715,20,True,0.023916,0.019758,"[18, 19, 20]","[48, 49, 50]","[48, 49, 50]"
3,1,curvature,4,50,0.000859,0.000802,50,45,0.002715,14,True,0.024411,0.018703,"[12, 13, 14]","[43, 44, 45]","[48, 49, 50]"
4,1,curvature,5,50,0.000649,0.001932,49,27,0.002715,14,True,0.025473,0.017833,"[12, 13, 14]","[25, 26, 27]","[48, 49, 50]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,2,vanilla,12,50,0.000475,0.000768,50,35,0.002715,14,True,0.022452,0.012347,"[12, 13, 14]","[33, 34, 35]","[48, 49, 50]"
156,2,vanilla,13,50,0.000596,0.001081,50,23,0.002715,13,True,0.021655,0.010379,"[11, 12, 13]","[21, 22, 23]","[48, 49, 50]"
157,2,vanilla,14,50,0.000536,0.000576,49,45,0.002715,11,True,0.021361,0.010238,"[9, 10, 11]","[43, 44, 45]","[48, 49, 50]"
158,2,vanilla,15,50,0.000514,0.000502,50,50,0.002715,16,True,0.021447,0.011788,"[14, 15, 16]","[48, 49, 50]","[48, 49, 50]"


In [35]:
metadata_df.to_csv("stats/metadata.csv")

In [9]:
# Summary across runs: mean/std of key quantities per (version, method)
agg_cols = [
    "best_train", "best_val",
    "best_train_ep", "best_val_ep",
    "target_ep", "std_train", "std_val"
]

summary_df = (
    metadata_df
    .groupby(["version", "method"], as_index=False)[agg_cols]
    .agg(["mean", "std", "count"])
)

# flatten MultiIndex columns
summary_df.columns = ["version", "method"] + [f"{c0}_{c1}" for (c0, c1) in summary_df.columns[2:]]
summary_df = summary_df.sort_values(["version", "best_val_mean"]).reset_index(drop=True)
summary_df


,version,method,best_train_mean,best_train_std,best_train_count,best_val_mean,best_val_std,best_val_count,best_train_ep_mean,best_train_ep_std,...,best_val_ep_count,target_ep_mean,target_ep_std,target_ep_count,std_train_mean,std_train_std,std_train_count,std_val_mean,std_val_std,std_val_count
0,1,stiefel,0.000557,0.000047,16,0.000874,0.000296,16,49.8125,0.403113,...,16,14.5625,4.647132,16,0.021949,0.000918,16,0.011261,0.001290,16
1,1,vanilla,0.000559,0.000068,16,0.000930,0.000387,16,49.6875,0.478714,...,16,14.5625,4.350766,16,0.022042,0.000811,16,0.011324,0.001427,16
2,1,curvature,0.000822,0.000097,16,0.001271,0.000370,16,49.6250,0.500000,...,16,14.5000,3.687818,16,0.024884,0.000608,16,0.020090,0.001797,16
3,1,operator_norm,0.001554,0.000117,16,0.001434,0.000497,16,49.5625,0.629153,...,16,19.6250,8.777813,16,0.024946,0.000541,16,0.020979,0.001427,16
4,1,stoch_iso,0.000787,0.000113,16,0.001648,0.000545,16,49.7500,0.447214,...,16,22.6250,11.412712,16,0.024758,0.000511,16,0.021241,0.001205,16


In [10]:
# -----------------
# Helpers: time grids + split segments from H5
# -----------------
def read_time_and_kind(f: h5py.File, field: str):
    if "coarse" in field:
        return f["t_coarse"][...], "coarse"
    return f["t_fine"][...], "fine"

def load_segment(f: h5py.File, kind: str, split: str):
    # Prefer explicit split datasets from the paper-aligned generator
    if "splits" in f:
        if kind == "fine":
            key = {"train":"splits/fine_ids_train",
                   "interp":"splits/fine_ids_interp",
                   "extrap":"splits/fine_ids_extrap"}[split]
        else:
            key = {"train":"splits/coarse_ids_train",
                   "interp":"splits/coarse_ids_val_loop",
                   "extrap":"splits/coarse_ids_val_loop"}[split]
        ids = np.asarray(f[key][...], dtype=np.int64)
        return int(ids[0]), int(ids[-1])

    # Fallback: infer from attrs iT1_*, iT2_*
    if kind == "fine":
        iT1 = int(f.attrs["iT1_fine"]); iT2 = int(f.attrs["iT2_fine"]); Nt = f["t_fine"].shape[0]
    else:
        iT1 = int(f.attrs["iT1_coarse"]); iT2 = int(f.attrs["iT2_coarse"]); Nt = f["t_coarse"].shape[0]

    if split == "train":  return 0, iT1
    if split == "interp": return iT1 + 1, iT2
    if split == "extrap": return iT2 + 1, Nt - 1
    raise ValueError(split)

with h5py.File(H5_PATH, "r") as f:
    t_eval_np, kind = read_time_and_kind(f, EVAL_FIELD)
    seg0, seg1 = load_segment(f, kind, SPLIT)
    U = f[f"{SPLIT}/{EVAL_FIELD}"]
    Nmu_all, Nt_all, H, W = U.shape

print("EVAL_FIELD:", EVAL_FIELD, "kind:", kind, "U:", (Nmu_all, Nt_all, H, W))
print("Segment:", SPLIT, "t ids:", (seg0, seg1), "len:", (seg1 - seg0 + 1))

EVAL_FIELD: u_fine kind: fine U: (200, 1001, 32, 32)
Segment: extrap t ids: (501, 1000) len: 500


In [11]:
# -----------------
# Normalization: compute umin/umax on (train μ) x (train-time) using u_coarse.
# Cached to norm_cache.json.
# -----------------
def get_or_compute_norm(h5_path: str, cache_path: str = "norm_cache.json"):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as g:
            d = json.load(g)
        if d.get("h5_path") == os.path.abspath(h5_path):
            return float(d["umin"]), float(d["umax"])

    with h5py.File(h5_path, "r") as f:
        train_idx = np.asarray(f["train_idx"][...], dtype=np.int64)
        if "splits" in f and "coarse_ids_train" in f["splits"]:
            train_t_ids = np.asarray(f["splits/coarse_ids_train"][...], dtype=np.int64)
        else:
            iT1 = int(f.attrs["iT1_coarse"])
            train_t_ids = np.arange(0, iT1 + 1, dtype=np.int64)

    ds = H5Snapshots(h5_path, split="train", field="u_coarse", mu_indices=train_idx, return_mu=False)
    umin, umax = ds.compute_train_minmax(field="u_coarse", mu_ids=train_idx, t_ids=train_t_ids, split="train")

    with open(cache_path, "w") as g:
        json.dump({"h5_path": os.path.abspath(h5_path), "umin": float(umin), "umax": float(umax)}, g)

    return float(umin), float(umax)

umin, umax = get_or_compute_norm(H5_PATH)
print("umin/umax:", umin, umax)

umin/umax: 0.0 3.20582914352417


In [12]:
# -----------------
# Sample evaluation windows (paired across methods)
# -----------------
def sample_windows(h5_path: str, split: str, field: str, horizon: int, n_mu: int, n_starts: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    with h5py.File(h5_path, "r") as f:
        U = f[f"{split}/{field}"]
        Nmu, Nt, *_ = U.shape
        t_np, kind = read_time_and_kind(f, field)
        seg0, seg1 = load_segment(f, kind, split)

    max_k0 = seg1 - horizon
    if max_k0 < seg0:
        raise ValueError(f"Segment too short for horizon={horizon}: seg=[{seg0},{seg1}]")

    mu_ids = rng.choice(np.arange(Nmu), size=min(n_mu, Nmu), replace=True)
    windows = []
    for mu in mu_ids:
        k0s = rng.integers(low=seg0, high=max_k0 + 1, size=n_starts)
        for k0 in k0s:
            windows.append((int(mu), int(k0)))
    return windows

windows = sample_windows(H5_PATH, SPLIT, EVAL_FIELD, HORIZON, N_MU, N_STARTS, seed=SEED)
print("Num windows:", len(windows), "example:", windows[:5])

Num windows: 1000 example: [(170, 669), (170, 831), (170, 633), (170, 601), (170, 833)]


In [13]:
# -----------------
# Load models
# -----------------
def load_models(methods, V: int, run: int, selection: str, device: str):
    out = {}
    for m in methods:
        enc = torch.load(f"output/AE_v{V}/{m}/enc_target.pt", map_location=device, weights_only=False)
        dec = torch.load(f"output/AE_v{V}/{m}/dec_target.pt", map_location=device, weights_only=False)
        ode = torch.load(f"output/NODE/{m}_v{V}/run_{run}/ode_{selection}.pt", map_location=device, weights_only=False)
        enc.eval(); dec.eval(); ode.eval()
        out[m] = (enc.to(device), dec.to(device), ode.to(device))
    return out



In [14]:
# -----------------
# Batched rollout + error metrics
# -----------------
def get_mu_tensor(mu_all_np, mu_ids, device):
    # mu_all_np: (Nmu, 3) float32 numpy
    return torch.as_tensor(mu_all_np[mu_ids], device=device, dtype=DTYPE)

def gather_u_windows(f: h5py.File, split: str, field: str, batch_windows, horizon: int):
    U = f[f"{split}/{field}"]
    out = []
    for (mu, k0) in batch_windows:
        out.append(np.asarray(U[mu, k0:k0+horizon+1], dtype=np.float32))  # (L+1,H,W)
    return np.stack(out, axis=0)  # (B,L+1,H,W)

def build_time_grid(t_np: np.ndarray, k0s: np.ndarray, horizon: int, device: str):
    """
    Exact absolute-time grid per window, required because the ODE uses absolute-time encoding.
    Returns t_grid: (B, L+1) torch float32.
    """
    offsets = np.arange(horizon + 1, dtype=np.int64)[None, :]  # (1,L+1)
    idx = k0s[:, None] + offsets                                # (B,L+1)
    t_grid_np = t_np[idx]                                       # (B,L+1)
    return torch.as_tensor(t_grid_np, device=device, dtype=DTYPE)

@torch.no_grad()
def rollout(enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method="rk2", drop_t0=True):
    """
    u_win_np: (B,L+1,H,W) physical
    t_grid:   (B,L+1) absolute time
    mu_t:     (B,nmu)
    """
    B, Lp1, H, W = u_win_np.shape
    device = mu_t.device

    u_true = torch.as_tensor(u_win_np, device=device, dtype=DTYPE).unsqueeze(2)  # (B,L+1,1,H,W)
    u0 = u_true[:, 0]  # (B,1,H,W)

    u0_n = norm_u(u0, umin, umax)
    z0 = enc(u0_n)

    z_pred = integrate_latent_eval(ode, z0, t_grid, mu_t, method=rk_method)   # (B,L+1,1,4,4)
    u_hat_n = dec(z_pred.reshape(-1, *z_pred.shape[2:])).reshape(B, Lp1, 1, H, W)
    u_hat = inv_norm_u(u_hat_n, umin, umax)

    # errors (physical space)
    diff = (u_hat - u_true).reshape(B, Lp1, -1)
    tru  = (u_true).reshape(B, Lp1, -1)

    num = torch.linalg.vector_norm(diff, dim=-1)  # (B,L+1)
    den = torch.linalg.vector_norm(tru,  dim=-1)  # (B,L+1)

    if drop_t0:
        num = num[:, 1:]
        den = den[:, 1:]
        diff = diff[:, 1:]

    rel = num / (den + 1e-8)
    mse = diff.pow(2).mean(dim=-1)

    metrics = {
        "abs_mean": num.mean(dim=1),
        "abs_max":  num.max(dim=1).values,
        "abs_fin":  num[:, -1],
        "rel_mean": rel.mean(dim=1),
        "rel_max":  rel.max(dim=1).values,
        "rel_fin":  rel[:, -1],
        "mse_fin":  mse[:, -1],
    }
    return metrics, z_pred, u_hat, u_true

In [15]:
# -----------------
# Intrinsic metrics
# -----------------
def dyn_jac_svals(ode, z, t_scalar, mu_row):
    with torch.enable_grad():
        z_flat = z.reshape(-1).detach().requires_grad_(True)

        def f_flat(z_flat_):
            zz = z_flat_.view(1, 1, 4, 4)
            out = ode(t_scalar.view(1), zz, mu_row.view(1, -1))
            return out.reshape(-1)

        J = torch.autograd.functional.jacobian(f_flat, z_flat, create_graph=False)
        return torch.linalg.svdvals(J).detach()


# -----------------
# Deterministic decoder gain (same probe every call)
# -----------------
def decoder_gain_power(dec, z, n_iter=5, probe_seed=0):
    """Approx spectral norm of J_dec at z via power iteration on J^T J, with deterministic init."""
    with torch.enable_grad():
        gen = torch.Generator(device=z.device)
        gen.manual_seed(int(probe_seed))

        v = torch.randn(z.shape, device=z.device, dtype=z.dtype, generator=gen)
        v = v / (v.flatten().norm() + 1e-12)

        for _ in range(n_iter):
            z0 = z.detach().requires_grad_(True)

            y = dec(z0).flatten()

            _, jv = torch.autograd.functional.jvp(
                lambda zz: dec(zz).flatten(),
                (z0,),
                (v,),
                create_graph=False
            )

            dot = (y * jv.detach()).sum()
            jt_jv = torch.autograd.grad(dot, z0, retain_graph=False, create_graph=False)[0]

            v = jt_jv.detach()
            v = v / (v.flatten().norm() + 1e-12)

        z0 = z.detach().requires_grad_(True)
        _, jv = torch.autograd.functional.jvp(
            lambda zz: dec(zz).flatten(),
            (z0,),
            (v,),
            create_graph=False
        )
        return jv.norm().detach()


@torch.no_grad()
def latent_truth(enc, u_true, umin, umax):
    B, Lp1 = u_true.shape[0], u_true.shape[1]
    u_n = norm_u(u_true.reshape(-1, *u_true.shape[2:]), umin, umax)
    z_true = enc(u_n).reshape(B, Lp1, 1, 4, 4)
    return z_true

def intrinsic_for_batch(enc, dec, ode, z_pred, u_true, t_grid, mu_t, steps_idx, dec_iters=5):
    rows = []
    z_true = latent_truth(enc, u_true, umin, umax)

    B = z_pred.shape[0]
    for b in range(B):
        mu_row = mu_t[b]
        for sidx in steps_idx:
            z = z_pred[b, sidx:sidx+1]           # (1,1,4,4)
            t_scalar = t_grid[b, sidx]

            svals = dyn_jac_svals(ode, z, t_scalar, mu_row)
            # gain  = decoder_gain_power(dec, z, n_iter=dec_iters)
            gain  = decoder_gain_power(dec, z, n_iter=dec_iters, probe_seed=int(sidx))
            zerr  = (z_pred[b, sidx] - z_true[b, sidx]).reshape(-1).norm()

            rows.append({
                "b": b,
                "step": int(sidx),
                "t": float(t_scalar.detach().cpu()),
                "dyn_sigma_max": float(svals.max().cpu()),
                "dyn_sigma_min": float(svals.min().cpu()),
                "dyn_cond": float((svals.max()/(svals.min()+1e-12)).cpu()),
                "dec_gain": float(gain.cpu()),
                "latent_err": float(zerr.detach().cpu()),
            })
    return pd.DataFrame(rows)

In [16]:
# -----------------
# Evaluate all methods (paired windows)
# -----------------
def eval_all(models, windows, split, field, horizon, batch_size, run_id=None):
    all_rows = []
    intr_rows = []

    steps_idx = np.linspace(0, horizon, num=INTR_STEPS).round().astype(int).tolist()

    # choose intrinsic windows by POSITION, not by (mu_id, k0) value
    intr_window_ids = set(range(min(INTR_MAX_WINDOWS, len(windows))))

    with h5py.File(H5_PATH, "r") as f:
        t_np, _ = read_time_and_kind(f, field)

        MU_ALL = None
        mu_key = f"{split}/mu"
        if mu_key in f:
            MU_ALL = np.asarray(f[mu_key][...], dtype=np.float32)

        for method, (enc, dec, ode) in models.items():
            print("Evaluating:", method)

            for i0 in range(0, len(windows), batch_size):
                batch_windows = windows[i0:i0+batch_size]
                batch_wids = np.arange(i0, i0 + len(batch_windows), dtype=np.int64)

                mu_ids = np.asarray([w[0] for w in batch_windows], dtype=np.int64)
                k0s    = np.asarray([w[1] for w in batch_windows], dtype=np.int64)

                u_win_np = gather_u_windows(f, split, field, batch_windows, horizon)
                t_grid = build_time_grid(t_np, k0s, horizon, DEVICE)

                mu_t = (
                    get_mu_tensor(MU_ALL, mu_ids, DEVICE)
                    if MU_ALL is not None
                    else torch.zeros((len(mu_ids), 0), device=DEVICE, dtype=DTYPE)
                )

                metrics, z_pred, u_hat, u_true = rollout(
                    enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method=RK_METHOD
                )

                for j, (mu_id, k0) in enumerate(batch_windows):
                    row = {
                        "method": method,
                        "split": split,
                        "field": field,
                        "horizon": horizon,
                        "window_id": int(batch_wids[j]),   # unique sampled-window id
                        "mu_id": int(mu_id),
                        "k0": int(k0),
                        "abs_mean": float(metrics["abs_mean"][j].cpu()),
                        "abs_max":  float(metrics["abs_max"][j].cpu()),
                        "abs_fin":  float(metrics["abs_fin"][j].cpu()),
                        "rel_mean": float(metrics["rel_mean"][j].cpu()),
                        "rel_max":  float(metrics["rel_max"][j].cpu()),
                        "rel_fin":  float(metrics["rel_fin"][j].cpu()),
                        "mse_fin":  float(metrics["mse_fin"][j].cpu()),
                    }
                    if run_id is not None:
                        row["run"] = int(run_id)
                    all_rows.append(row)

                # intrinsic subset by window_id, not tuple membership
                keep = [idx for idx, wid in enumerate(batch_wids) if int(wid) in intr_window_ids]
                if keep:
                    df_i = intrinsic_for_batch(
                        enc, dec, ode,
                        z_pred[keep], u_true[keep], t_grid[keep], mu_t[keep],
                        steps_idx=steps_idx, dec_iters=DEC_GAIN_ITERS
                    )
                    df_i["method"] = method
                    df_i["split"] = split
                    df_i["field"] = field
                    df_i["horizon"] = horizon
                    df_i["window_id"] = df_i["b"].map(lambda bb: int(batch_wids[keep[bb]]))
                    df_i["mu_id"]     = df_i["b"].map(lambda bb: int(batch_windows[keep[bb]][0]))
                    df_i["k0"]        = df_i["b"].map(lambda bb: int(batch_windows[keep[bb]][1]))
                    if run_id is not None:
                        df_i["run"] = int(run_id)
                    df_i.drop(columns=["b"], inplace=True)
                    intr_rows.append(df_i)

    df_roll = pd.DataFrame(all_rows)
    df_intr = pd.concat(intr_rows, ignore_index=True) if intr_rows else pd.DataFrame()
    return df_roll, df_intr


# -----------------
# Summary tables
# -----------------
def summarize(df, cols):
    g = df.groupby("method")[cols]
    out = g.agg(["mean","std","count"])
    for c in cols:
        out[(c,"se")] = out[(c,"std")] / np.sqrt(out[(c,"count")].clip(lower=1))
    return out.sort_values((cols[0],"mean"))


# -----------------
# Paired comparisons vs vanilla
# -----------------
from scipy.stats import wilcoxon

def paired_vs_baseline(df, metric: str, baseline: str = "vanilla"):
    # pair by sampled-window identity; include run if available
    keys = ["window_id"]
    if "run" in df.columns:
        keys = ["run"] + keys

    wide = df.pivot_table(index=keys, columns="method", values=metric, aggfunc="first")

    if baseline not in wide.columns:
        raise ValueError(f"baseline {baseline} not present in {list(wide.columns)}")

    deltas = wide.sub(wide[baseline], axis=0).drop(columns=[baseline], errors="ignore")

    rows = []
    for m in deltas.columns:
        x = deltas[m].dropna().to_numpy()
        if len(x) >= 5:
            p_less = wilcoxon(x, alternative="less").pvalue
            p_two  = wilcoxon(x, alternative="two-sided").pvalue
        else:
            p_less = p_two = np.nan

        rows.append({
            "metric": metric,
            "method": m,
            "n": len(x),
            "delta_mean": float(np.mean(x)) if len(x) else np.nan,
            "delta_median": float(np.median(x)) if len(x) else np.nan,
            "p_less": p_less,
            "p_two_sided": p_two,
        })

    return pd.DataFrame(rows).sort_values("p_less")


for seed in NODE_RUNS:
    print(f"RUN {seed} [{SELECTION}] ----------------------------------------")

    models = load_models(METHODS, V, seed, SELECTION, DEVICE)

    df_roll, df_intr = eval_all(models, windows, SPLIT, EVAL_FIELD, HORIZON, BATCH, run_id=seed)
    print(df_roll.head())
    print("roll rows:", len(df_roll), "intr rows:", len(df_intr))

    summary = summarize(df_roll, ["rel_mean","rel_max","rel_fin","abs_fin","mse_fin"])
    print(f"[{seed}] summary:")
    print(summary)

    paired = paired_vs_baseline(df_roll, "rel_mean")
    print(f"{seed} [{SELECTION}] paired:")
    print(paired)

    os.makedirs(f"stats/{SELECTION}", exist_ok=True)

    df_roll.to_csv(f"stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv", index=False)
    df_intr.to_csv(f"stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv", index=False)

    summary.to_csv(f"stats/{SELECTION}/summary_H{HORIZON}_v{V}_{seed}.csv")
    paired.to_csv(f"stats/{SELECTION}/paired_H{HORIZON}_v{V}_{seed}.csv", index=False)

    print("----------------------------------------\n")

RUN 1 [best_swa] ----------------------------------------
Evaluating: vanilla
Evaluating: stoch_iso
Evaluating: operator_norm
Evaluating: curvature
Evaluating: stiefel
    method   split   field  horizon  window_id  mu_id   k0  abs_mean  \
0  vanilla  extrap  u_fine       80          0    170  669  4.273297   
1  vanilla  extrap  u_fine       80          1    170  831  5.732998   
2  vanilla  extrap  u_fine       80          2    170  633  5.255528   
3  vanilla  extrap  u_fine       80          3    170  601  7.758431   
4  vanilla  extrap  u_fine       80          4    170  833  5.680832   

     abs_max   abs_fin  rel_mean   rel_max   rel_fin   mse_fin  run  
0   6.804361  3.085994  0.113796  0.187087  0.088570  0.009300    1  
1   6.944651  6.599882  0.121685  0.191938  0.188033  0.042538    1  
2   6.542908  5.912664  0.113386  0.180469  0.169010  0.034140    1  
3  10.488599  7.597692  0.136003  0.171939  0.156789  0.056372    1  
4   6.969243  6.460940  0.122726  0.192479  0.184

In [17]:
df_intr = pd.DataFrame()
df_roll = pd.DataFrame()

for seed in range(1, 4 + 1):
    df_i = pd.read_csv(f"stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv")
    df_r = pd.read_csv(f"stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv")

    # backward-compatible: inject run if old files do not contain it
    if "run" not in df_i.columns:
        df_i["run"] = seed
    if "run" not in df_r.columns:
        df_r["run"] = seed

    df_intr = pd.concat([df_intr, df_i], ignore_index=True)
    df_roll = pd.concat([df_roll, df_r], ignore_index=True)

In [18]:
df_intr.head(10)

,step,t,dyn_sigma_max,dyn_sigma_min,dyn_cond,dec_gain,latent_err,method,split,field,horizon,window_id,mu_id,k0,run
0,0,21.017256,2.243443,0.007438,301.619141,3.462635,0.000101,vanilla,extrap,u_fine,80,0,170,669,1
1,16,21.519909,1.745073,0.001344,1298.750732,3.417148,0.831997,vanilla,extrap,u_fine,80,0,170,669,1
2,32,22.022564,1.674767,0.013991,119.704323,3.383747,1.473803,vanilla,extrap,u_fine,80,0,170,669,1
3,48,22.525219,2.311243,0.016267,142.085907,2.783578,1.149245,vanilla,extrap,u_fine,80,0,170,669,1
4,64,23.027874,3.075174,0.003028,1015.665527,2.988534,0.911488,vanilla,extrap,u_fine,80,0,170,669,1
5,80,23.530529,2.035406,0.020395,99.798264,3.343315,0.755642,vanilla,extrap,u_fine,80,0,170,669,1
6,0,26.106634,3.166565,0.048665,65.068390,3.485919,0.000114,vanilla,extrap,u_fine,80,1,170,831,1
7,16,26.609289,3.097548,0.036098,85.808998,3.433368,1.172641,vanilla,extrap,u_fine,80,1,170,831,1
8,32,27.111944,2.399360,0.012144,197.575226,3.485292,1.408380,vanilla,extrap,u_fine,80,1,170,831,1
9,48,27.614599,2.001656,0.010276,194.794113,3.354074,1.488418,vanilla,extrap,u_fine,80,1,170,831,1


In [19]:
# -----------------
# Intrinsic metric summaries
# -----------------
if len(df_intr):
    intr_summary = df_intr.groupby("method")[["dyn_sigma_max","dyn_cond","dec_gain","latent_err"]].agg(["mean","std","count"])
    intr_summary
else:
    print("No intrinsic rows computed.")

In [20]:
intr_summary

dyn_sigma_max                     dyn_cond                      \
                       mean       std count         mean           std count   
method                                                                         
curvature          4.164903  1.087440  1920  2001.523107  23155.417362  1920   
operator_norm      4.309017  1.132002  1920  1394.975331   5464.800270  1920   
stiefel            3.214086  0.856643  1920   831.494468   6382.608159  1920   
stoch_iso          4.401234  1.206583  1920  1726.312061   8378.888566  1920   
vanilla            2.981250  0.742150  1920   736.255997   5682.225976  1920   

               dec_gain                 latent_err                  
                   mean       std count       mean       std count  
method                                                              
curvature      1.606547  0.012287  1920   2.525016  1.770402  1920  
operator_norm  1.028088  0.027279  1920   2.939335  2.327526  1920  
stiefel        4.646441  0.194503  1920   0.722832  0.482455  1920  
stoch_iso      1.000420  0.001491  1920   2.987489  2.356064  1920  
vanilla        3.327971  0.180456  1920   0.871019  0.624010  1920

In [21]:
# intr_summary.to_csv(f"stats/{SELECTION}/intr_summary_H{HORIZON}_v{V}_{RUN}.csv")

In [22]:
import numpy as np
import pandas as pd

df = df_intr.copy()
df = df.drop(columns=[c for c in ["Unnamed: 0"] if c in df.columns])

# Make sure numeric
num_cols = ["step","t","dyn_sigma_max","dyn_sigma_min","dyn_cond","dec_gain","latent_err"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Logs (clip avoids -inf if something hits 0)
eps = 1e-12
df["log10_dyn_cond"] = np.log10(np.clip(df["dyn_cond"].to_numpy(), eps, None))
df["log10_sigma_max"] = np.log10(np.clip(df["dyn_sigma_max"].to_numpy(), eps, None))
df["log10_sigma_min"] = np.log10(np.clip(df["dyn_sigma_min"].to_numpy(), eps, None))
df["log10_dec_gain"]  = np.log10(np.clip(df["dec_gain"].to_numpy(), eps, None))

# Tail / “catastrophe” rates (tune thresholds after looking at quantiles)
df["cond_gt_1e4"]  = df["dyn_cond"] > 1e4
df["cond_gt_1e5"]  = df["dyn_cond"] > 1e5
df["sigmax_gt_10"] = df["dyn_sigma_max"] > 10


In [23]:
qs = [0.5, 0.9, 0.95, 0.99, 0.999]

def qcols(s):
    q = s.quantile(qs)
    q.index = [f"q{int(p*1000):03d}" for p in qs]  # q500 q900 q950 q990 q999
    return q

summary_method = (
    df.groupby("method")
      .apply(lambda g: pd.concat([
          qcols(g["log10_dyn_cond"]).add_prefix("log10_dyn_cond_"),
          qcols(g["latent_err"]).add_prefix("latent_err_"),
          qcols(g["dyn_sigma_max"]).add_prefix("sigmax_"),
          g[["cond_gt_1e4","cond_gt_1e5","sigmax_gt_10"]].mean().add_prefix("rate_"),
      ]))
      .reset_index()
)

summary_method


,method,log10_dyn_cond_q500,log10_dyn_cond_q900,log10_dyn_cond_q950,log10_dyn_cond_q990,log10_dyn_cond_q999,latent_err_q500,latent_err_q900,latent_err_q950,latent_err_q990,latent_err_q999,sigmax_q500,sigmax_q900,sigmax_q950,sigmax_q990,sigmax_q999,rate_cond_gt_1e4,rate_cond_gt_1e5,rate_sigmax_gt_10
0,curvature,2.498475,3.241822,3.629636,4.336836,5.353862,2.319296,4.882542,5.779823,7.374800,8.942828,4.023780,5.586537,6.075949,7.293323,8.956307,0.021875,0.002083,0.0
1,operator_norm,2.642144,3.387527,3.630796,4.215907,4.828213,2.465056,5.992690,7.105951,10.328127,13.776625,4.168164,5.794253,6.415287,7.566209,8.668167,0.019792,0.000521,0.0
2,stiefel,2.143764,2.907198,3.266918,4.124506,5.009027,0.706289,1.303242,1.550726,2.090633,2.647156,3.086874,4.331803,4.757448,5.952117,6.872764,0.012500,0.001563,0.0
3,stoch_iso,2.640754,3.412505,3.717037,4.338268,5.005342,2.542359,6.075707,7.279190,10.767034,13.565942,4.230485,6.041337,6.623408,7.801902,9.101630,0.025521,0.001042,0.0
4,vanilla,2.281287,3.001916,3.270008,3.976615,4.474492,0.826127,1.602434,1.955346,2.795482,3.937552,2.923557,3.955012,4.295951,5.140784,5.670199,0.007812,0.000521,0.0


In [24]:
# one row per (run, method, sampled window) after aggregating over intrinsic steps
run_keys = ["run", "method", "split", "field", "horizon", "window_id"]

run_stats = (
    df.groupby(run_keys)
      .agg(
          mu_id=("mu_id", "first"),
          k0=("k0", "first"),
          logcond_med=("log10_dyn_cond", "median"),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          cond_rate_1e5=("cond_gt_1e5", "mean"),
          sigmax_med=("dyn_sigma_max", "median"),
          laterr_med=("latent_err", "median"),
      )
      .reset_index()
)

# aggregate across runs/windows, still separated by method
method_run_summary = (
    run_stats.groupby(["method", "split", "field", "horizon"])
             .agg(["mean", "std", "median"])
)

method_run_summary

run                  window_id  \
                                    mean       std median      mean   
method        split  field  horizon                                   
curvature     extrap u_fine 80       2.5  1.119785    2.5      39.5   
operator_norm extrap u_fine 80       2.5  1.119785    2.5      39.5   
stiefel       extrap u_fine 80       2.5  1.119785    2.5      39.5   
stoch_iso     extrap u_fine 80       2.5  1.119785    2.5      39.5   
vanilla       extrap u_fine 80       2.5  1.119785    2.5      39.5   

                                                        mu_id             \
                                           std median    mean        std   
method        split  field  horizon                                        
curvature     extrap u_fine 80       23.128373   39.5  67.375  57.038594   
operator_norm extrap u_fine 80       23.128373   39.5  67.375  57.038594   
stiefel       extrap u_fine 80       23.128373   39.5  67.375  57.038594   
stoch_iso     extrap u_fine 80       23.128373   39.5  67.375  57.038594   
vanilla       extrap u_fine 80       23.128373   39.5  67.375  57.038594   

                                                 k0  ... logcond_p99  \
                                    median     mean  ...      median   
method        split  field  horizon                  ...               
curvature     extrap u_fine 80        57.0  725.375  ...    3.182668   
operator_norm extrap u_fine 80        57.0  725.375  ...    3.308793   
stiefel       extrap u_fine 80        57.0  725.375  ...    2.831654   
stoch_iso     extrap u_fine 80        57.0  725.375  ...    3.386094   
vanilla       extrap u_fine 80        57.0  725.375  ...    2.909302   

                                    cond_rate_1e5                  sigmax_med  \
                                             mean       std median       mean   
method        split  field  horizon                                             
curvature     extrap u_fine 80           0.002083  0.018546    0.0   4.078481   
operator_norm extrap u_fine 80           0.000521  0.009317    0.0   4.219549   
stiefel       extrap u_fine 80           0.001563  0.016087    0.0   3.148333   
stoch_iso     extrap u_fine 80           0.001042  0.013155    0.0   4.318980   
vanilla       extrap u_fine 80           0.000521  0.009317    0.0   2.936785   

                                                        laterr_med            \
                                          std    median       mean       std   
method        split  field  horizon                                            
curvature     extrap u_fine 80       0.604167  4.054821   2.830144  1.196247   
operator_norm extrap u_fine 80       0.614668  4.180828   3.311580  1.755807   
stiefel       extrap u_fine 80       0.553897  3.089172   0.808529  0.290606   
stoch_iso     extrap u_fine 80       0.707115  4.274277   3.357413  1.765247   
vanilla       extrap u_fine 80       0.466110  2.973563   0.976152  0.406708   

                                               
                                       median  
method        split  field  horizon            
curvature     extrap u_fine 80       2.567353  
operator_norm extrap u_fine 80       2.980864  
stiefel       extrap u_fine 80       0.757477  
stoch_iso     extrap u_fine 80       2.997915  
vanilla       extrap u_fine 80       0.887015  

[5 rows x 27 columns]

In [25]:
time_profile = (
    df.groupby(["method","split","field","horizon","step"])
      .agg(
          logcond_med=("log10_dyn_cond","median"),
          logcond_p95=("log10_dyn_cond", lambda s: s.quantile(0.95)),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          n=("log10_dyn_cond","size"),
      )
      .reset_index()
)

time_profile


,method,split,field,horizon,step,logcond_med,logcond_p95,logcond_p99,n
0,curvature,extrap,u_fine,80,0,2.419630,3.478082,4.324609,320
1,curvature,extrap,u_fine,80,16,2.505114,3.537923,4.340855,320
2,curvature,extrap,u_fine,80,32,2.492458,3.529265,3.899606,320
3,curvature,extrap,u_fine,80,48,2.525015,3.767623,4.618399,320
4,curvature,extrap,u_fine,80,64,2.480602,3.704145,4.140874,320
5,curvature,extrap,u_fine,80,80,2.542614,3.528104,4.345192,320
6,operator_norm,extrap,u_fine,80,0,2.639633,3.661113,4.322105,320
7,operator_norm,extrap,u_fine,80,16,2.621895,3.741963,4.325121,320
8,operator_norm,extrap,u_fine,80,32,2.622343,3.512444,3.986977,320
9,operator_norm,extrap,u_fine,80,48,2.693725,3.613045,4.094744,320


In [26]:
intr_per_run = run_stats.copy()

merge_keys = ["run", "method", "split", "field", "horizon", "window_id"]

roll_unique = (
    df_roll[merge_keys + ["mu_id", "k0", "rel_mean", "mse_fin"]]
    .drop_duplicates(subset=merge_keys)
)

merged = intr_per_run.merge(
    roll_unique,
    on=merge_keys,
    how="inner",
)

merged[["logcond_med", "logcond_p99", "cond_rate_1e5", "laterr_med", "rel_mean", "mse_fin"]].corr()

,logcond_med,logcond_p99,cond_rate_1e5,laterr_med,rel_mean,mse_fin
logcond_med,1.000000,0.553494,0.045122,0.349266,0.113146,0.061369
logcond_p99,0.553494,1.000000,0.300838,0.200873,0.065534,0.002003
cond_rate_1e5,0.045122,0.300838,1.000000,-0.002745,-0.018478,-0.004690
laterr_med,0.349266,0.200873,-0.002745,1.000000,0.799077,0.639062
rel_mean,0.113146,0.065534,-0.018478,0.799077,1.000000,0.745092
mse_fin,0.061369,0.002003,-0.004690,0.639062,0.745092,1.000000
